In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
pd.set_option('display.max_columns', None)


In [2]:
df = pd.read_parquet("../data/processed/spy_processed.parquet")
df.columns

Index(['symbol', 'expiration', 'strike', 'right', 'timestamp', 'bid', 'ask',
       'delta', 'theta', 'vega', 'rho', 'epsilon', 'lambda', 'implied_vol',
       'iv_error', 'underlying_timestamp', 'underlying_price', 'spy_close',
       'spy_open', 'log_return_from_open', 'ttm_min', 'log_return', 'd1',
       'gamma', 'open_interest', 'dex', 'gex'],
      dtype='str')

In [3]:
aggregate = pd.read_parquet("../data/processed/spy_aggregate.parquet")
aggregate.columns

Index(['timestamp', 'net_dex', 'net_gex', 'underlying_price', 'net_gex_norm',
       'net_dex_norm'],
      dtype='str')

In [6]:
# ATM IV: mean IV of call and put at the strike closest to spot for each timestamp
df2 = df.assign(dist=(df["strike"] - df["underlying_price"]).abs())
min_dist = df2.groupby("timestamp")["dist"].transform("min")
atm_iv = (
    df2[df2["dist"] == min_dist]
    .groupby("timestamp")["implied_vol"]
    .mean()
    .rename("atm_iv")
    .reset_index()
)
aggregate = aggregate.merge(atm_iv, on="timestamp", how="left")
aggregate

,timestamp,net_dex,net_gex,underlying_price,net_gex_norm,net_dex_norm,atm_iv
0,2026-05-06 09:45:00-04:00,-6.482183e+09,3.514162e+09,729.15,4.819533e+06,-8890054.53,0.16965
1,2026-05-06 10:00:00-04:00,-6.673169e+09,3.507746e+09,729.66,4.807371e+06,-9145586.46,0.16185
2,2026-05-06 10:15:00-04:00,-6.249268e+09,3.465909e+09,730.08,4.747301e+06,-8559702.83,0.16135
3,2026-05-06 10:30:00-04:00,-7.254773e+09,3.651162e+09,730.03,5.001387e+06,-9937636.19,0.16025
4,2026-05-06 10:45:00-04:00,-6.235425e+09,2.971337e+09,731.85,4.060035e+06,-8520086.28,0.14300
...,...,...,...,...,...,...,...
245,2026-05-19 14:45:00-04:00,1.357632e+09,-1.324910e+09,735.84,-1.800541e+06,1845010.04,0.23725
246,2026-05-19 15:00:00-04:00,1.671626e+09,-1.950403e+09,734.42,-2.655705e+06,2276117.74,0.27925
247,2026-05-19 15:15:00-04:00,1.285933e+09,-1.285623e+09,736.02,-1.746722e+06,1747143.67,0.22770
248,2026-05-19 15:30:00-04:00,2.363966e+09,-2.252909e+09,734.77,-3.066142e+06,3217287.20,0.20060


In [13]:
df[(df["timestamp"] == "2026-05-06 10:45:00-04:00") & (np.abs(df['strike'] - df['underlying_price']) < 0.5)]

,symbol,expiration,strike,right,timestamp,bid,ask,delta,theta,vega,rho,epsilon,lambda,implied_vol,iv_error,underlying_timestamp,underlying_price,spy_close,spy_open,log_return_from_open,ttm_min,log_return,d1,gamma,open_interest,dex,gex
1075,SPY,2026-05-06,732.0,CALL,2026-05-06 10:45:00-04:00,0.93,0.94,0.4792,-2.3050,7.1378,0.2096,-0.2101,377.4797,0.1391,-0.0009,2026-05-06 10:45:00-04:00,731.85,733.77,728.19,0.005014,315.0,0.00262,-0.051440,0.159867,3227,-1.131717e+08,2.763133e+08
1100,SPY,2026-05-06,732.0,PUT,2026-05-06 10:45:00-04:00,1.12,1.13,-0.5195,-2.3605,7.1389,-0.2285,0.2279,-339.8163,0.1469,-0.0008,2026-05-06 10:45:00-04:00,731.85,733.77,728.19,0.005014,315.0,0.00262,-0.048523,0.151401,44,1.672863e+06,-3.567994e+06
